# CNN Classifier with Data Augmentation — Pneumonia vs Healthy

Train `SimpleCNN` on NIH Chest X-ray dataset **with** data augmentation on the training set:  
`RandomRotation` · `RandomHorizontalFlip` · `ColorJitter (brightness)` · `RandomResizedCrop`

Uses `pos_weight` on `BCEWithLogitsLoss` to deal with unbalanced classes.  
Val/Test sets are **not** augmented (standard evaluation).

**Reported metrics:** Accuracy and AUC-ROC on test dataset.

## 1. Setup

In [ ]:
import sys
import os
import json
from pathlib import Path

# Add utils and models to path
project_root = Path("../").resolve()
sys.path.insert(0, str(project_root / "utils"))
sys.path.insert(0, str(project_root / "models"))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from classifier import SimpleCNN
from device import DEVICE
from metrics import print_metrics, plot_roc_curve

print(f"Device: {DEVICE}")

## 2. Hyperparameters

In [ ]:
IMG_SIZE    = (128, 128)
BATCH_SIZE  = 64
LR          = 3e-4
NUM_EPOCHS  = 20
DROPOUT     = 0.3
SEED        = 42

CHECKPOINT_PATH = project_root / "models" / "best_classifier_augmented.pt"
RESULTS_DIR     = project_root / "results"
RESULTS_DIR.mkdir(exist_ok=True)

## 3. Load Dataset (from cache)

In [ ]:
DATA_CACHE_DIR = project_root / "data" / "processed"
train_dataset = torch.load(DATA_CACHE_DIR / "train_dataset.pt", weights_only=False)
val_dataset   = torch.load(DATA_CACHE_DIR / "val_dataset.pt",   weights_only=False)
test_dataset  = torch.load(DATA_CACHE_DIR / "test_dataset.pt",  weights_only=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## 4. Data Augmentation

Augmentation is applied **only** to the training set via a thin `AugmentedDataset` wrapper.  
The underlying `PyTorchDataset` stores PIL images, so torchvision transforms are applied  
directly on the PIL image before converting to tensor.

| Transform | Parameters | Rationale |
|---|---|---|
| `RandomRotation` | ±15° | Small in-plane rotations are realistic for X-rays |
| `RandomHorizontalFlip` | p=0.5 | Left/right flip is valid for chest anatomy |
| `ColorJitter` (brightness) | factor 0.2 | Simulates exposure variation |
| `RandomResizedCrop` | scale 85–100% | Simulates slight zoom/positioning differences |

In [ ]:
aug_transform = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2),
    transforms.RandomResizedCrop(size=IMG_SIZE, scale=(0.85, 1.0)),
])


class AugmentedDataset(Dataset):
    """Wraps a PyTorchDataset and applies a transform to training images."""

    def __init__(self, base_dataset, transform):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img = self.base.images[idx].convert('L')  # grayscale PIL
        if self.transform:
            img = self.transform(img)
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.from_numpy(img_array).unsqueeze(0)  # (1, H, W)
        return img_tensor, self.base.labels[idx], self.base.metadata[idx]


aug_train_dataset = AugmentedDataset(train_dataset, aug_transform)
print(f"Augmented training set size: {len(aug_train_dataset)}")

### 4.1 Visualize augmented examples

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(SEED)

# Pick one sample and show it augmented 6 times
sample_idx = 0
original_img = train_dataset.images[sample_idx].convert('L')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(original_img, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flat[1:]):
    aug_img = aug_transform(original_img)
    ax.imshow(aug_img, cmap='gray')
    ax.set_title(f'Augmented #{i+1}')
    ax.axis('off')

fig.suptitle('Data Augmentation Examples (grayscale chest X-ray)', fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'augmentation_examples.png', dpi=150)
plt.show()

## 5. DataLoaders

In [ ]:
# Compute pos_weight = num_healthy / num_pneumonia in the training set
train_labels = train_dataset.labels[:, 1]  # index 1 = pneumonia
n_pos = train_labels.sum().item()
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)

print(f"Train — Healthy: {int(n_neg)} | Pneumonia: {int(n_pos)}")
print(f"pos_weight: {pos_weight.item():.2f}")

train_loader = DataLoader(aug_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 6. Model, Loss, Optimizer

In [ ]:
torch.manual_seed(SEED)

model     = SimpleCNN(dropout_rate=DROPOUT).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

print(model)

## 7. Training Loop

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    """Run one epoch. If optimizer is None, runs in eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_labels, all_scores = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels, _ in loader:
            images = images.to(device)                         # (B, 1, H, W)
            y = labels[:, 1].unsqueeze(1).float().to(device)   # pneumonia score

            logits = model(images)                             # (B, 1)
            loss = criterion(logits, y)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_labels.extend(y.cpu().squeeze(1).tolist())
            all_scores.extend(torch.sigmoid(logits).cpu().squeeze(1).tolist())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_labels), np.array(all_scores)

In [ ]:
from metrics import compute_roc_auc
from tqdm.notebook import tqdm
import time

best_val_auc = 0.0
history = {"train_loss": [], "train_auc": [], "val_loss": [], "val_auc": []}

pbar = tqdm(range(1, NUM_EPOCHS + 1), desc="Training", unit="epoch")
for epoch in pbar:
    start_time = time.time()
    train_loss, train_y, train_sc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_y, val_sc       = run_epoch(val_loader,   model, criterion)
    train_auc = compute_roc_auc(train_y, train_sc)
    val_auc   = compute_roc_auc(val_y, val_sc)
    scheduler.step(val_auc)

    history["train_loss"].append(train_loss)
    history["train_auc"].append(train_auc)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        flag = " ← best"
    else:
        flag = ""

    elapsed_time = time.time() - start_time
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} "
          f"| elapsed_time={elapsed_time:.2f}s "
          f"| train_loss={train_loss:.4f} "
          f"| train_auc={train_auc:.4f} "
          f"| val_loss={val_loss:.4f} "
          f"| val_auc={val_auc:.4f}{flag}")

print(f"\nBest val AUC: {best_val_auc:.4f}")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("BCEWithLogitsLoss (with augmentation)")
axes[0].legend()

axes[1].plot(epochs, history["train_auc"], label="Train")
axes[1].plot(epochs, history["val_auc"],   label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC-ROC")
axes[1].set_title("AUC-ROC (with augmentation)")
axes[1].legend()

fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves_augmented.png", dpi=150)
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

_, test_y, test_scores = run_epoch(test_loader, model, criterion)

print("=== Test Set Metrics ===")
print_metrics(test_y, test_scores)

In [ ]:
fig = plot_roc_curve(test_y, test_scores, save_path=RESULTS_DIR / "roc_curve_augmented.png")
plt.show()

## 10. Save Results

In [ ]:
from metrics import compute_accuracy, compute_roc_auc

results = {
    "model": "SimpleCNN",
    "augmentation": {
        "RandomRotation": {"degrees": 15},
        "RandomHorizontalFlip": {"p": 0.5},
        "ColorJitter": {"brightness": 0.2},
        "RandomResizedCrop": {"scale": [0.85, 1.0]},
    },
    "img_size": list(IMG_SIZE),
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout": DROPOUT,
    "seed": SEED,
    "pos_weight": round(pos_weight.item(), 4),
    "best_val_auc": round(best_val_auc, 4),
    "test_accuracy": round(compute_accuracy(test_y, test_scores), 4),
    "test_auc_roc":  round(compute_roc_auc(test_y, test_scores), 4),
    "history": history,
}

results_path = RESULTS_DIR / "augmented_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print(json.dumps({k: v for k, v in results.items() if k != 'history'}, indent=2))